# TrackNet ball tracker — Colab training

Heatmap tracker for the tiny/fast cricket ball (3 frames → ball heatmap, uses motion).

**Before you start:** Runtime → Change runtime type → **GPU (T4)**.

Upload two zips when asked: `tracknet_code.zip` (the .py files) and `dataset.zip` (packaged frames + manifest). The setup cell handles either zip layout automatically.

## 1. Environment

In [ ]:
!nvidia-smi -L
!pip -q install opencv-python-headless
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 2. Upload + auto-normalise (handles both zip layouts)

In [ ]:
from google.colab import files
import zipfile, os, shutil
print('Upload tracknet_code.zip and dataset.zip (you can select both):')
up = files.upload()
for z in ['tracknet_code.zip', 'dataset.zip']:
    if os.path.exists(z):
        zipfile.ZipFile(z).extractall('.'); print('extracted', z)

PY = ['model.py', 'dataset.py', 'train.py', 'infer.py']
os.makedirs('tracknet', exist_ok=True)
# Layout 2 (files in root) -> move them into tracknet/. Layout 1 (already in tracknet/) -> left as-is.
for name in PY:
    if os.path.exists(name) and not os.path.exists(f'tracknet/{name}'):
        shutil.move(name, f'tracknet/{name}')
open('tracknet/__init__.py', 'a').close()
# dataset may extract to ./dataset or ./tracknet/dataset -> normalise to ./dataset
if not os.path.isdir('dataset') and os.path.isdir('tracknet/dataset'):
    shutil.move('tracknet/dataset', 'dataset')

have = [f for f in PY if os.path.exists(f'tracknet/{f}')]
print('py files in tracknet/:', have)
print('dataset frames dir:', os.path.isdir('dataset/frames'), '| manifest:', os.path.exists('dataset/manifest.csv'))
assert len(have) == 4 and os.path.exists('dataset/manifest.csv'), 'setup incomplete — check the two uploads'
print('OK — ready to train')

## 3. Train

Prints **held-out peak-recall** each epoch (recall on 6 clips it never trains on) — the honest number to compare vs the YOLO pipeline's ~39%. Saves `tracknet_best.pt`.

In [ ]:
import sys; sys.path.insert(0, '.')
!python -m tracknet.train --data dataset --epochs 40 --batch 8 \
    --holdout bowling3,Test13,bowling_5,bowling_18,bowling_47,bowling_99 \
    --out tracknet_best.pt

## 4. Download the trained weights

In [ ]:
from google.colab import files
files.download('tracknet_best.pt')  # then locally: python tracknet/infer.py --video <clip> --weights tracknet_best.pt